# Chapter 5: 선형 회귀 (Linear Regression)

이 노트북은 머신러닝의 핵심 알고리즘인 **선형 회귀**를 단계별로 학습합니다.

| 섹션 | 내용 |
|------|------|
| 5.3 | 경사 하강법 (Gradient Descent) 직접 구현 |
| 5.4 | 사이킷런 LinearRegression으로 보스턴 주택 가격 예측 |
| 5.5 | 다항 회귀 (Polynomial Regression) 및 과적합/과소적합 |
| 5.6 | 정규화 회귀 (Ridge, Lasso, ElasticNet) |

---

## 5.3 Gradient Descent (경사 하강법)

### 개념 정리

**경사 하강법**은 비용 함수(Cost Function)를 최소화하기 위해 가중치(w)를 반복적으로 업데이트하는 최적화 알고리즘입니다.

핵심 아이디어:
- 현재 위치에서 **기울기(gradient)의 반대 방향**으로 조금씩 이동
- **학습률(learning_rate)**: 한 번에 이동하는 보폭 크기
- 이 과정을 반복하면 비용 함수의 **최솟값**에 수렴

목표: `y = 4X + 6` 형태의 선형 관계를 데이터로부터 학습 (w1=4, w0=6 근사)

**실제값을 Y=4X+6 시뮬레이션하는 데이터 값 생성**

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

# 재현성을 위해 랜덤 시드 고정
np.random.seed(0)

# y = 4X + 6 식을 근사(w1=4, w0=6). random 값은 Noise를 위해 만듬
# X: 0~2 사이의 균일 분포에서 100개 샘플 (100x1 행렬)
X = 2 * np.random.rand(100,1)
# y: 실제 관계식 + 표준정규분포 노이즈 추가
y = 6 + 4 * X + np.random.randn(100,1)

# X, y 데이터 셋 scatter plot으로 시각화
plt.scatter(X, y)

In [ ]:
# 데이터 형태 확인: X와 y 모두 (100, 1) 형태의 2D 배열
X.shape, y.shape

**w0과 w1의 값을 최소화 할 수 있도록 업데이트 수행하는 함수 생성**

* 예측 배열 y_pred는 np.dot(X, w1.T) + w0 임
100개의 데이터 X(1,2,...,100)이 있다면 예측값은 w0 + X(1)*w1 + X(2)*w1 +..+ X(100)*w1이며, 이는 입력 배열 X와 w1 배열의 내적임.
* 새로운 w1과 w0를 update함
![](./image01.png)

### 가중치 업데이트 수식

MSE 비용함수의 편미분을 이용한 업데이트 규칙:

$$w_1 \leftarrow w_1 - \frac{2}{N} \cdot \text{lr} \cdot X^T(y_{pred} - y)$$
$$w_0 \leftarrow w_0 - \frac{2}{N} \cdot \text{lr} \cdot \mathbf{1}^T(y_{pred} - y)$$

In [ ]:
# w1 과 w0 를 업데이트 할 w1_update, w0_update를 반환. 
def get_weight_updates(w1, w0, X, y, learning_rate=0.01):
    N = len(y)  # 데이터 개수 (100)
    
    # 먼저 w1_update, w0_update를 각각 w1, w0의 shape와 동일한 크기를 가진 0 값으로 초기화
    w1_update = np.zeros_like(w1)
    w0_update = np.zeros_like(w0)
    
    # 현재 w1, w0로 예측값 계산: y_pred = X·w1 + w0
    y_pred = np.dot(X, w1.T) + w0
    # 실제값과 예측값의 오차 계산 (잔차: residual)
    diff = y - y_pred
         
    # w0_update를 dot 행렬 연산으로 구하기 위해 모두 1값을 가진 행렬 생성
    # (절편 w0는 모든 샘플에 동일하게 적용되므로 1로 이루어진 벡터 필요)
    w0_factors = np.ones((N,1))

    # w1과 w0을 업데이트할 w1_update와 w0_update 계산
    # 음수 부호: 기울기의 반대 방향으로 이동 (하강)
    # (2/N): MSE 미분 시 나오는 상수
    w1_update = -(2/N)*learning_rate*(np.dot(X.T, diff))
    w0_update = -(2/N)*learning_rate*(np.dot(w0_factors.T, diff))    
    
    return w1_update, w0_update

In [ ]:
# 함수 동작 테스트: w0, w1을 0으로 초기화한 뒤 첫 번째 업데이트 값 확인
w0 = np.zeros((1,1))
w1 = np.zeros((1,1))

# 초기 예측값: 모두 0 (w0=0, w1=0이므로)
y_pred = np.dot(X, w1.T) + w0
diff = y - y_pred
print(diff.shape)  # (100, 1) 확인

# 수동으로 업데이트 값 계산
w0_factors = np.ones((100,1))
w1_update = -(2/100)*0.01*(np.dot(X.T, diff))
w0_update = -(2/100)*0.01*(np.dot(w0_factors.T, diff))   
print(w1_update.shape, w0_update.shape)  # 각각 (1,1) 형태
w1, w0  # 현재 가중치 (아직 0)

**반복적으로 경사 하강법을 이용하여 get_weigth_updates()를 호출하여 w1과 w0를 업데이트 하는 함수 생성**

### 전체 배치 경사 하강법 (Batch Gradient Descent)

매 iteration마다 **전체 데이터셋**을 사용하여 기울기를 계산하고 가중치를 업데이트합니다.

In [ ]:
# 입력 인자 iters로 주어진 횟수만큼 반복적으로 w1과 w0를 업데이트 적용함.
def gradient_descent_steps(X, y, iters=10000):
    # w0와 w1을 모두 0으로 초기화. (학습 시작점)
    w0 = np.zeros((1,1))
    w1 = np.zeros((1,1))
    
    # 인자로 주어진 iters 만큼 반복적으로 get_weight_updates() 호출하여 w1, w0 업데이트 수행.
    for ind in range(iters):
        # 현재 w1, w0에서의 업데이트 방향과 크기 계산
        w1_update, w0_update = get_weight_updates(w1, w0, X, y, learning_rate=0.01)
        # 업데이트 적용: 기울기 반대 방향으로 이동
        w1 = w1 - w1_update
        w0 = w0 - w0_update
              
    return w1, w0

**예측 오차 비용을 계산을 수행하는 함수 생성 및 경사 하강법 수행**

### 비용 함수 (Cost Function)

**MSE (Mean Squared Error)**: 예측값과 실제값의 차이를 제곱한 평균

$$\text{MSE} = \frac{1}{N}\sum_{i=1}^{N}(y_i - \hat{y}_i)^2$$

값이 작을수록 모델의 예측이 실제값에 가깝다는 의미

In [ ]:
def get_cost(y, y_pred):
    """MSE(Mean Squared Error) 비용 함수 계산"""
    N = len(y) 
    cost = np.sum(np.square(y - y_pred)) / N  # 잔차 제곱의 평균
    return cost

# 1000번 반복으로 경사 하강법 실행
w1, w0 = gradient_descent_steps(X, y, iters=1000)
print("w1:{0:.3f} w0:{1:.3f}".format(w1[0,0], w0[0,0]))  # 이상적: w1≈4, w0≈6

# 학습된 가중치로 예측
y_pred = w1[0,0] * X + w0
print('Gradient Descent Total Cost:{0:.4f}'.format(get_cost(y, y_pred)))

In [ ]:
# 학습 결과 시각화
# 산점도: 실제 데이터 포인트
plt.scatter(X, y)
# 직선: 경사 하강법으로 학습한 회귀선
plt.plot(X, y_pred)

**미니 배치 확률적 경사 하강법을 이용한 최적 비용함수 도출**

### 미니 배치 확률적 경사 하강법 (Mini-batch SGD)

| 방법 | 특징 |
|------|------|
| Batch GD | 전체 데이터 사용 → 안정적이나 느림 |
| SGD | 1개 샘플만 사용 → 빠르나 불안정 |
| **Mini-batch SGD** | **일부 샘플(batch) 사용 → 속도와 안정성 균형** |

매 iteration마다 전체 데이터에서 `batch_size`개를 **랜덤 추출**하여 학습

In [ ]:
def stochastic_gradient_descent_steps(X, y, batch_size=10, iters=1000):
    """미니 배치 확률적 경사 하강법
    
    Args:
        X: 입력 피처 배열
        y: 타겟 배열
        batch_size: 매 iteration에서 사용할 샘플 수 (기본값: 10)
        iters: 반복 횟수 (기본값: 1000)
    """
    w0 = np.zeros((1,1))  # 절편 초기화
    w1 = np.zeros((1,1))  # 기울기 초기화
    prev_cost = 100000    # 이전 비용 초기값 (매우 큰 값)
    iter_index = 0
    
    for ind in range(iters):
        np.random.seed(ind)  # iteration마다 다른 시드로 다양한 배치 샘플링
        
        # 전체 X, y 데이터에서 랜덤하게 batch_size만큼 데이터 추출하여 sample_X, sample_y로 저장
        # permutation: 0~N-1 인덱스를 랜덤하게 섞어 앞의 batch_size개 선택
        stochastic_random_index = np.random.permutation(X.shape[0])
        sample_X = X[stochastic_random_index[0:batch_size]]
        sample_y = y[stochastic_random_index[0:batch_size]]
        
        # 랜덤하게 batch_size만큼 추출된 데이터 기반으로 w1_update, w0_update 계산 후 업데이트
        w1_update, w0_update = get_weight_updates(w1, w0, sample_X, sample_y, learning_rate=0.01)
        w1 = w1 - w1_update
        w0 = w0 - w0_update
    
    return w1, w0

In [ ]:
# 미니 배치 SGD 실행 및 결과 비교
w1, w0 = stochastic_gradient_descent_steps(X, y, iters=1000)
print("w1:", round(w1[0,0], 3), "w0:", round(w0[0,0], 3))  # Batch GD 결과와 비교

y_pred = w1[0,0] * X + w0
print('Stochastic Gradient Descent Total Cost:{0:.4f}'.format(get_cost(y, y_pred)))
# 일반적으로 Batch GD보다 약간 높은 비용, 하지만 실제 대규모 데이터에서는 훨씬 빠름

---
## 5.4 사이킷런 LinearRegression을 이용한 보스턴 주택 가격 예측

### 개념 정리

**사이킷런(Scikit-learn)**의 `LinearRegression`은 **OLS(Ordinary Least Squares, 최소제곱법)**을 사용합니다.

- 경사 하강법처럼 반복 계산 없이 **수식으로 직접 최적해**를 구함
- $w = (X^TX)^{-1}X^Ty$

### 보스턴 주택 가격 데이터셋

- 506개 샘플, 13개의 주택/지역 관련 피처
- 타겟: 주택 중앙 가격 (단위: 천 달러)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from scipy import stats
from sklearn.datasets import load_boston
import warnings
warnings.filterwarnings('ignore')  # 사이킷런 1.2 부터는 보스턴 주택가격 데이터가 없어진다는 warning 메시지 출력 제거
%matplotlib inline

# boston 데이타셋 로드
boston = load_boston()

# boston 데이타셋 DataFrame 변환
bostonDF = pd.DataFrame(boston.data, columns=boston.feature_names)

# boston dataset의 target array는 주택 가격임. 이를 PRICE 컬럼으로 DataFrame에 추가함.
bostonDF['PRICE'] = boston.target
print('Boston 데이타셋 크기 :', bostonDF.shape)  # (506, 14) 예상
bostonDF.head()

### 보스턴 데이터셋 피처 설명

* **CRIM**: 지역별 범죄 발생률  
* **ZN**: 25,000평방피트를 초과하는 거주 지역의 비율
* **NDUS**: 비상업 지역 넓이 비율
* **CHAS**: 찰스강에 대한 더미 변수(강의 경계에 위치한 경우는 1, 아니면 0)
* **NOX**: 일산화질소 농도
* **RM**: 거주할 수 있는 방 개수
* **AGE**: 1940년 이전에 건축된 소유 주택의 비율
* **DIS**: 5개 주요 고용센터까지의 가중 거리
* **RAD**: 고속도로 접근 용이도
* **TAX**: 10,000달러당 재산세율
* **PTRATIO**: 지역의 교사와 학생 수 비율
* **B**: 지역의 흑인 거주 비율
* **LSTAT**: 하위 계층의 비율
* **MEDV**: 본인 소유의 주택 가격(중앙값) ← **타겟 변수**

### 각 피처와 주택 가격의 관계 시각화

`seaborn.regplot`으로 산점도 + 회귀선을 동시에 표시하여 각 피처가 PRICE에 미치는 영향을 직관적으로 확인합니다.

In [ ]:
# 2개의 행과 4개의 열을 가진 subplots를 이용. axs는 4x2개의 ax를 가짐.
fig, axs = plt.subplots(figsize=(16, 8), ncols=4, nrows=2)

# 주택 가격과 관계가 높을 것으로 예상되는 8개 피처 선택
lm_features = ['RM', 'ZN', 'INDUS', 'NOX', 'AGE', 'PTRATIO', 'LSTAT', 'RAD']

for i, feature in enumerate(lm_features):
    row = int(i / 4)   # 0~3: 첫 번째 행, 4~7: 두 번째 행
    col = i % 4        # 열 위치 (0~3 순환)
    # 시본의 regplot을 이용해 산점도와 선형 회귀 직선을 함께 표현
    # 양의 기울기 → 가격과 양의 상관, 음의 기울기 → 음의 상관
    sns.regplot(x=feature, y='PRICE', data=bostonDF, ax=axs[row][col])

fig1 = plt.gcf()
fig1.savefig('p322_boston.tif', format='tif', dpi=300, bbox_inches='tight')

**학습과 테스트 데이터 세트로 분리하고 학습/예측/평가 수행**

### 모델 학습 파이프라인

1. **데이터 분리**: train/test split (7:3)
2. **모델 학습**: `lr.fit(X_train, y_train)`
3. **예측**: `lr.predict(X_test)`
4. **평가**: MSE, RMSE, R² Score

**평가 지표 설명:**
- **MSE**: 오차의 제곱 평균 (단위: 달러²)
- **RMSE**: MSE의 제곱근 (단위: 달러, 해석이 쉬움)
- **R² Score**: 0~1 사이 값, 1에 가까울수록 모델 설명력 높음

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# 타겟(y)과 피처(X) 분리
y_target = bostonDF['PRICE']
X_data = bostonDF.drop(['PRICE'], axis=1, inplace=False)

# 훈련(70%) / 테스트(30%) 분리, random_state로 재현성 보장
X_train, X_test, y_train, y_test = train_test_split(X_data, y_target, test_size=0.3, random_state=156)

# Linear Regression OLS로 학습/예측/평가 수행.
lr = LinearRegression()
lr.fit(X_train, y_train)     # 학습: 최적 w 계산
y_preds = lr.predict(X_test)  # 예측

# 평가 지표 계산
mse = mean_squared_error(y_test, y_preds)
rmse = np.sqrt(mse)

print('MSE : {0:.3f} , RMSE : {1:.3F}'.format(mse, rmse))
print('Variance score : {0:.3f}'.format(r2_score(y_test, y_preds)))

In [ ]:
# 학습된 모델의 파라미터 확인
# intercept_: 절편(w0), 즉 모든 피처가 0일 때의 예측 가격
print('절편 값:', lr.intercept_)
# coef_: 각 피처의 회귀 계수(w), 해당 피처가 1 증가할 때 가격 변화량
print('회귀 계수값:', np.round(lr.coef_, 1))

In [ ]:
# 회귀 계수를 큰 값 순으로 정렬하기 위해 Series로 생성. index가 칼럼명에 유의
# 양수: 해당 피처가 증가하면 주택 가격 상승
# 음수: 해당 피처가 증가하면 주택 가격 하락
coeff = pd.Series(data=np.round(lr.coef_, 1), index=X_data.columns)
coeff.sort_values(ascending=False)

### 교차 검증 (Cross-Validation)

**K-Fold 교차 검증**: 데이터를 K개 폴드로 나누고, 각 폴드를 한 번씩 테스트 세트로 사용
- 단순 train/test split보다 **신뢰도 높은 성능 평가** 가능
- 특정 분리에 의한 운 좋은/나쁜 결과를 평균화

> **주의**: `cross_val_score(scoring="neg_mean_squared_error")`는 음수 MSE를 반환하므로, RMSE 계산 시 `-1`을 곱해서 양수로 변환 필요

In [ ]:
from sklearn.model_selection import cross_val_score

y_target = bostonDF['PRICE']
X_data = bostonDF.drop(['PRICE'], axis=1, inplace=False)
lr = LinearRegression()

# cross_val_score( )로 5 Fold 셋으로 MSE 를 구한 뒤 이를 기반으로 다시  RMSE 구함.
# scoring='neg_mean_squared_error': MSE의 음수값 반환 (사이킷런 규칙: 높을수록 좋은 방향)
neg_mse_scores = cross_val_score(lr, X_data, y_target, scoring="neg_mean_squared_error", cv=5)
# 음수 MSE → 양수로 변환 후 제곱근 취해 RMSE 계산
rmse_scores = np.sqrt(-1 * neg_mse_scores)
avg_rmse = np.mean(rmse_scores)

# cross_val_score(scoring="neg_mean_squared_error")로 반환된 값은 모두 음수
print(' 5 folds 의 개별 Negative MSE scores: ', np.round(neg_mse_scores, 2))
print(' 5 folds 의 개별 RMSE scores : ', np.round(rmse_scores, 2))
print(' 5 folds 의 평균 RMSE : {0:.3f} '.format(avg_rmse))

---
## 5-5. Polynomial Regression과 오버피팅/언더피팅 이해

### 개념 정리

**다항 회귀(Polynomial Regression)**는 피처에 다항식 변환을 적용하여 비선형 관계를 선형 모델로 학습합니다.

예: 입력 피처 $[x_1, x_2]$ → degree=2 변환 → $[1, x_1, x_2, x_1^2, x_1x_2, x_2^2]$

| 차수 | 특징 |
|------|------|
| 낮음 (1차) | 과소적합(Underfitting): 너무 단순하여 패턴을 학습 못 함 |
| 적절 (4차) | 실제 패턴에 근접하게 학습 |
| 높음 (15차) | 과적합(Overfitting): 훈련 데이터의 노이즈까지 학습 |

### Polynomial Regression 이해

PolynomialFeatures 클래스로 다항식 변환

![](./image02.png)

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
import numpy as np

# 다항식으로 변환한 단항식 생성, [[0,1],[2,3]]의 2X2 행렬 생성
X = np.arange(4).reshape(2, 2)  # [[0,1], [2,3]]
print('일차 단항식 계수 feature:\n', X)

# degree=2 → [1, x1, x2, x1², x1*x2, x2²] 형태로 변환
# degree = 2 인 2차 다항식으로 변환하기 위해 PolynomialFeatures를 이용하여 변환
poly = PolynomialFeatures(degree=2)
poly.fit(X)
poly_ftr = poly.transform(X)
print('변환된 2차 다항식 계수 feature:\n', poly_ftr)
# 원본 2개 피처 → 변환 후 6개 피처: [1, x0, x1, x0^2, x0*x1, x1^2]

3차 다항식 결정값을 구하는 함수 polynomial_func(X) 생성. 즉 회귀식은 결정값 y = 1+ 2x_1 + 3x_1^2 + 4x_2^3

In [ ]:
def polynomial_func(X):
    """3차 다항식: y = 1 + 2*x1 + 3*x1² + 4*x2³"""
    y = 1 + 2*X[:, 0] + 3*X[:, 0]**2 + 4*X[:, 1]**3
    print(X[:, 0])  # 첫 번째 피처 (x1)
    print(X[:, 1])  # 두 번째 피처 (x2)
    return y

X = np.arange(0, 4).reshape(2, 2)  # [[0,1], [2,3]]

print('일차 단항식 계수 feature: \n', X)
y = polynomial_func(X)
print('삼차 다항식 결정값: \n', y)

3차 다항식 계수의 피처값과 3차 다항식 결정값으로 학습

In [ ]:
# 3 차 다항식 변환
# fit_transform: fit + transform을 한 번에 수행
poly_ftr = PolynomialFeatures(degree=3).fit_transform(X)
print('3차 다항식 계수 feature: \n', poly_ftr)

# Linear Regression에 3차 다항식 계수 feature와 3차 다항식 결정값으로 학습 후 회귀 계수 확인
# PolynomialFeatures로 변환한 피처로 LinearRegression을 학습하면 다항 회귀와 동일
model = LinearRegression()
model.fit(poly_ftr, y)
print('Polynomial 회귀 계수\n', np.round(model.coef_, 2))
print('Polynomial 회귀 Shape :', model.coef_.shape)

**사이킷런 파이프라인(Pipeline)을 이용하여 3차 다항회귀 학습**  

사이킷런의 Pipeline 객체는 Feature 엔지니어링 변환과 모델 학습/예측을 순차적으로 결합해줍니다.

### Pipeline 사용 장점

- 전처리 → 학습을 **한 번의 `.fit()` 호출**로 처리
- 코드가 깔끔하고 **데이터 누수(Data Leakage) 방지**
- `cross_val_score` 등과 함께 사용 시 각 fold마다 자동으로 전처리 적용

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
import numpy as np

def polynomial_func(X):
    y = 1 + 2*X[:, 0] + 3*X[:, 0]**2 + 4*X[:, 1]**3
    return y

# Pipeline 객체로 Streamline 하게 Polynomial Feature변환과 Linear Regression을 연결
# 순서대로: (이름, 객체) 튜플의 리스트
model = Pipeline([('poly', PolynomialFeatures(degree=3)),      # Step 1: 3차 다항식 변환
                  ('linear', LinearRegression())])              # Step 2: 선형 회귀 학습

X = np.arange(4).reshape(2, 2)
y = polynomial_func(X)

# fit 호출 시 poly 변환 → linear 학습이 자동으로 순서대로 실행됨
model = model.fit(X, y)

# named_steps['이름']으로 Pipeline 내부 특정 단계의 객체에 접근
print('Polynomial 회귀 계수\n', np.round(model.named_steps['linear'].coef_, 2))

### 다항 회귀를 이용한 과소적합 및 과적합 이해

**cosine 곡선에 약간의 Noise 변동값을 더하여 실제값 곡선을 만듬**

차수(degree)에 따른 모델 복잡도와 성능 변화를 시각적으로 확인합니다:

- **degree=1**: 직선 → 실제 cos 곡선을 전혀 학습 못함 (과소적합)
- **degree=4**: 실제 곡선과 유사하게 학습 (적절)
- **degree=15**: 훈련 데이터의 노이즈까지 학습 (과적합)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score
%matplotlib inline

# 임의의 값으로 구성된 X값에 대해 코사인 변환 값을 반환.
# 이것이 우리가 학습해야 할 '실제(True)' 함수
def true_fun(X):
    return np.cos(1.5 * np.pi * X)

# X는 0부터 1까지 30개의 임의의 값을 순서대로 샘플링한 데이터입니다.
np.random.seed(0)
n_samples = 30
X = np.sort(np.random.rand(n_samples))  # 0~1 사이 균일분포에서 30개 추출 후 정렬

# y 값은 코사인 기반의 true_fun()에서 약간의 노이즈 변동 값을 더한 값입니다.
# 실제 데이터에서 측정 오차나 불확실성을 시뮬레이션
y = true_fun(X) + np.random.randn(n_samples) * 0.1

In [ ]:
# 데이터 분포 확인 (cos 형태에 노이즈가 추가된 모습)
plt.scatter(X, y)

In [ ]:
plt.figure(figsize=(14, 5))
degrees = [1, 4, 15]  # 비교할 차수: 과소적합 / 적절 / 과적합

# 다항 회귀의 차수(degree)를 1, 4, 15로 각각 변화시키면서 비교합니다.
for i in range(len(degrees)):
    ax = plt.subplot(1, len(degrees), i + 1)
    plt.setp(ax, xticks=(), yticks=())  # 축 눈금 제거 (깔끔한 시각화)
    
    # 개별 degree별로 Polynomial 변환합니다.
    polynomial_features = PolynomialFeatures(degree=degrees[i], include_bias=False)
    linear_regression = LinearRegression()
    # Pipeline으로 변환+학습 연결
    pipeline = Pipeline([("polynomial_features", polynomial_features),
                         ("linear_regression", linear_regression)])
    # X를 2D 배열로 reshape (Pipeline 입력 형식)
    pipeline.fit(X.reshape(-1, 1), y)
    
    # 교차 검증으로 다항 회귀를 평가합니다. (10-fold)
    scores = cross_val_score(pipeline, X.reshape(-1, 1), y,
                             scoring="neg_mean_squared_error", cv=10)
    # Pipeline을 구성하는 세부 객체를 접근하는 named_steps['객체명']을 이용해 회귀계수 추출
    coefficients = pipeline.named_steps['linear_regression'].coef_
    print('\nDegree {0} 회귀 계수는 {1} 입니다.'.format(degrees[i], np.round(coefficients, 2)))
    print('Degree {0} MSE 는 {1} 입니다.'.format(degrees[i], -1*np.mean(scores)))
          
    # 0 부터 1까지 테스트 데이터 세트를 100개로 나눠 예측을 수행합니다.
    # 촘촘한 X값으로 예측하여 부드러운 예측 곡선 생성
    X_test = np.linspace(0, 1, 100)
    # 예측값 곡선 (모델이 학습한 패턴)
    plt.plot(X_test, pipeline.predict(X_test[:, np.newaxis]), label="Model")
    # 실제 값 곡선 (노이즈 없는 순수 cos 함수)
    plt.plot(X_test, true_fun(X_test), '--', label="True function")
    # 학습에 사용된 실제 데이터 포인트
    plt.scatter(X, y, edgecolor='b', s=20, label="Samples")
    plt.xlabel("x"); plt.ylabel("y")
    plt.xlim((0, 1)); plt.ylim((-2, 2))
    plt.legend(loc="best")
    plt.title("Degree {}\nMSE = {:.2e}(+/- {:.2e})".format(
        degrees[i], -scores.mean(), scores.std()))
    
plt.show()
# 결과 해석:
# degree=1: 직선으로 cos 곡선 표현 불가 → 높은 MSE (과소적합)
# degree=4: cos 곡선에 근접 → 낮은 MSE (최적)
# degree=15: 훈련 데이터에 구불구불하게 맞춤 → 매우 높은 MSE (과적합)

---
## 5-6. Regularized Linear Models – Ridge, Lasso

### 개념 정리: 정규화 (Regularization)

**정규화**는 모델의 복잡도를 제어하여 **과적합을 방지**하는 기법입니다.
회귀 계수(가중치)가 너무 커지지 않도록 **비용 함수에 패널티 항**을 추가합니다.

| 모델 | 패널티 항 | 특징 |
|------|-----------|------|
| **Ridge** | $\alpha \sum w_i^2$ (L2) | 계수를 0에 가깝게 줄임, 모든 피처 유지 |
| **Lasso** | $\alpha \sum |w_i|$ (L1) | 일부 계수를 정확히 0으로 만듦 (피처 선택 효과) |
| **ElasticNet** | L1 + L2 혼합 | Ridge와 Lasso의 장점 결합 |

**alpha**: 정규화 강도. 클수록 계수가 더 많이 줄어듦

### Regularized Linear Model - Ridge Regression

In [ ]:
# 앞의 LinearRegression예제에서 분할한 feature 데이터 셋인 X_data과 Target 데이터 셋인 Y_target 데이터셋을 그대로 이용
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score

# boston 데이타셋 로드
boston = load_boston()

# boston 데이타셋 DataFrame 변환
bostonDF = pd.DataFrame(boston.data, columns=boston.feature_names)

# boston dataset의 target array는 주택 가격임. 이를 PRICE 컬럼으로 DataFrame에 추가함.
bostonDF['PRICE'] = boston.target

y_target = bostonDF['PRICE']
X_data = bostonDF.drop(['PRICE'], axis=1, inplace=False)

# Ridge 회귀: alpha=10으로 L2 정규화 적용
ridge = Ridge(alpha=10)
neg_mse_scores = cross_val_score(ridge, X_data, y_target,
                                 scoring="neg_mean_squared_error", cv=5)
rmse_scores = np.sqrt(-1 * neg_mse_scores)
avg_rmse = np.mean(rmse_scores)
print(' 5 folds 의 개별 Negative MSE scores: ', np.round(neg_mse_scores, 3))
print(' 5 folds 의 개별 RMSE scores : ', np.round(rmse_scores, 3))
print(' 5 folds 의 평균 RMSE : {0:.3f} '.format(avg_rmse))

**alpha값을 0 , 0.1 , 1 , 10 , 100 으로 변경하면서 RMSE 측정**

alpha가 커질수록 정규화 강도가 세지고 계수가 줄어들어 모델이 단순해집니다.
최적 alpha는 검증 성능(RMSE)이 가장 낮은 값으로 선택합니다.

In [ ]:
# 릿지에 사용될 alpha 파라미터의 값을 정의
alphas = [0, 0.1, 1, 10, 100]

# alphas list 값을 반복하면서 alpha에 따른 평균 rmse를 구함.
# alpha=0: 정규화 없음 (일반 LinearRegression과 동일)
# alpha=100: 강한 정규화 (계수가 매우 작아짐)
for alpha in alphas:
    ridge = Ridge(alpha=alpha)
    
    # cross_val_score를 이용해 5 폴드의 평균 RMSE를 계산
    neg_mse_scores = cross_val_score(ridge, X_data, y_target,
                                     scoring="neg_mean_squared_error", cv=5)
    avg_rmse = np.mean(np.sqrt(-1 * neg_mse_scores))
    print('alpha {0} 일 때 5 folds 의 평균 RMSE : {1:.3f} '.format(alpha, avg_rmse))

**각 alpha에 따른 회귀 계수 값을 시각화. 각 alpha값 별로 plt.subplots로 맷플롯립 축 생성**

alpha가 커질수록 회귀 계수들이 0에 수렴하는 것을 확인할 수 있습니다.

In [ ]:
# 각 alpha에 따른 회귀 계수 값을 시각화하기 위해 5개의 열로 된 맷플롯립 축 생성
fig, axs = plt.subplots(figsize=(18, 6), nrows=1, ncols=5)
# 각 alpha에 따른 회귀 계수 값을 데이터로 저장하기 위한 DataFrame 생성
coeff_df = pd.DataFrame()

# alphas 리스트 값을 차례로 입력해 회귀 계수 값 시각화 및 데이터 저장. pos는 axis의 위치 지정
for pos, alpha in enumerate(alphas):
    ridge = Ridge(alpha=alpha)
    ridge.fit(X_data, y_target)  # 전체 데이터로 학습
    
    # alpha에 따른 피처별 회귀 계수를 Series로 변환하고 이를 DataFrame의 컬럼으로 추가.
    coeff = pd.Series(data=ridge.coef_, index=X_data.columns)
    colname = 'alpha:' + str(alpha)
    coeff_df[colname] = coeff  # DataFrame에 alpha별 계수 저장
    
    # 막대 그래프로 각 alpha 값에서의 회귀 계수를 시각화. 회귀 계수값이 높은 순으로 표현
    coeff = coeff.sort_values(ascending=False)
    axs[pos].set_title(colname)
    axs[pos].set_xlim(-3, 6)  # 모든 서브플롯 동일 x축 범위로 비교 용이
    sns.barplot(x=coeff.values, y=coeff.index, ax=axs[pos])

# for 문 바깥에서 맷플롯립의 show 호출 및 alpha에 따른 피처별 회귀 계수를 DataFrame으로 표시
plt.show()
# 시각화 관찰 포인트:
# alpha=0(왼쪽): 계수 크기가 다양함
# alpha=100(오른쪽): 모든 계수가 0에 수렴 (정규화 효과)

**alpha 값에 따른 컬럼별 회귀계수 출력**

In [ ]:
# alpha=0일 때의 계수 기준으로 내림차순 정렬하여 표시
# alpha가 커질수록 각 계수들이 어떻게 변하는지 숫자로 확인 가능
ridge_alphas = [0, 0.1, 1, 10, 100]
sort_column = 'alpha:' + str(ridge_alphas[0])
coeff_df.sort_values(by=sort_column, ascending=False)

### 라쏘 회귀 (Lasso Regression)

**Lasso**는 L1 정규화를 사용하여 일부 회귀 계수를 정확히 **0으로 만들어** 중요하지 않은 피처를 자동으로 제거하는 **피처 선택** 효과가 있습니다.

> Ridge vs Lasso: Ridge는 계수를 0에 **가깝게** 줄이지만, Lasso는 **정확히 0**으로 만들 수 있음

In [ ]:
from sklearn.linear_model import Lasso, ElasticNet

# alpha값에 따른 회귀 모델의 폴드 평균 RMSE를 출력하고 회귀 계수값들을 DataFrame으로 반환
def get_linear_reg_eval(model_name, params=None, X_data_n=None, y_target_n=None,
                        verbose=True, return_coeff=True):
    """Ridge, Lasso, ElasticNet 모델의 성능을 alpha별로 평가하는 범용 함수
    
    Args:
        model_name: 'Ridge', 'Lasso', 'ElasticNet' 중 하나
        params: 평가할 alpha 값의 리스트
        X_data_n: 피처 데이터
        y_target_n: 타겟 데이터
        verbose: True면 모델명 헤더 출력
        return_coeff: True면 계수 DataFrame 반환
    """
    coeff_df = pd.DataFrame()
    if verbose:
        print('####### ', model_name, '#######')
    
    for param in params:
        # 모델명에 따라 해당 정규화 모델 생성
        if model_name == 'Ridge': model = Ridge(alpha=param)
        elif model_name == 'Lasso': model = Lasso(alpha=param)
        elif model_name == 'ElasticNet': model = ElasticNet(alpha=param, l1_ratio=0.7)  # L1 70%, L2 30%
        
        # 5-fold 교차 검증으로 평균 RMSE 계산
        neg_mse_scores = cross_val_score(model, X_data_n,
                                         y_target_n, scoring="neg_mean_squared_error", cv=5)
        avg_rmse = np.mean(np.sqrt(-1 * neg_mse_scores))
        print('alpha {0}일 때 5 폴드 세트의 평균 RMSE: {1:.3f} '.format(param, avg_rmse))
        
        # cross_val_score는 evaluation metric만 반환하므로 모델을 다시 학습하여 회귀 계수 추출
        # 전체 데이터로 재학습하여 피처별 계수 확인
        model.fit(X_data_n, y_target_n)
        if return_coeff:
            # alpha에 따른 피처별 회귀 계수를 Series로 변환하고 이를 DataFrame의 컬럼으로 추가.
            coeff = pd.Series(data=model.coef_, index=X_data_n.columns)
            colname = 'alpha:' + str(param)
            coeff_df[colname] = coeff
    
    return coeff_df
# end of get_linear_regre_eval

In [ ]:
# 라쏘에 사용될 alpha 파라미터의 값들을 정의하고 get_linear_reg_eval() 함수 호출
# Ridge보다 작은 alpha값 사용 (Lasso는 같은 alpha에서 더 강한 정규화 효과)
lasso_alphas = [0.07, 0.1, 0.5, 1, 3]
coeff_lasso_df = get_linear_reg_eval('Lasso', params=lasso_alphas,
                                     X_data_n=X_data, y_target_n=y_target)

In [ ]:
# 반환된 coeff_lasso_df를 첫번째 컬럼순으로 내림차순 정렬하여 회귀계수 DataFrame출력
# 관찰 포인트: alpha가 커질수록 0이 되는 계수(피처)가 늘어남 → Lasso의 피처 선택 효과
sort_column = 'alpha:' + str(lasso_alphas[0])
coeff_lasso_df.sort_values(by=sort_column, ascending=False)

### 엘라스틱넷 회귀 (ElasticNet Regression)

**ElasticNet** = Ridge(L2) + Lasso(L1)의 결합

$$\text{비용함수} = \text{MSE} + \alpha \cdot [\text{l1\_ratio} \cdot \sum|w_i| + (1-\text{l1\_ratio}) \cdot \sum w_i^2]$$

- `l1_ratio=1.0`: 순수 Lasso
- `l1_ratio=0.0`: 순수 Ridge
- `l1_ratio=0.7`: L1 70% + L2 30% 혼합 (본 예제)

In [ ]:
# 엘라스틱넷에 사용될 alpha 파라미터의 값들을 정의하고 get_linear_reg_eval() 함수 호출
# l1_ratio는 0.7로 고정 (L1:L2 = 7:3 비율)
elastic_alphas = [0.07, 0.1, 0.5, 1, 3]
coeff_elastic_df = get_linear_reg_eval('ElasticNet', params=elastic_alphas,
                                       X_data_n=X_data, y_target_n=y_target)

In [ ]:
# 반환된 coeff_elastic_df를 첫번째 컬럼순으로 내림차순 정렬하여 회귀계수 DataFrame출력
# Lasso와 Ridge의 중간 특성: 일부 계수는 0, 나머지는 작은 값으로 유지
sort_column = 'alpha:' + str(elastic_alphas[0])
coeff_elastic_df.sort_values(by=sort_column, ascending=False)

### 선형 회귀 모델을 위한 데이터 변환

선형 회귀는 피처가 **정규 분포**에 가까울수록 성능이 좋습니다.
데이터 변환 방법을 비교하여 최적의 전처리 방법을 찾습니다.

| 변환 방법 | 설명 | 적합한 경우 |
|-----------|------|------------|
| StandardScaler | 평균=0, 표준편차=1로 정규화 | 대부분의 경우 |
| MinMaxScaler | 0~1 범위로 정규화 | 이상치가 없는 경우 |
| Log 변환 | `log(x+1)` 적용 | 왜도가 큰 분포 |
| + Polynomial | 다항식 피처 추가 | 비선형 관계 |

각 방법을 Ridge 회귀와 조합하여 RMSE를 비교합니다.

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler, PolynomialFeatures

# method는 표준 정규 분포 변환(Standard), 최대값/최소값 정규화(MinMax), 로그변환(Log) 결정
# p_degree는 다향식 특성을 추가할 때 적용. p_degree는 2이상 부여하지 않음.
def get_scaled_data(method='None', p_degree=None, input_data=None):
    """데이터 스케일링 및 다항식 변환을 수행하는 함수
    
    Args:
        method: 스케일링 방법 ('Standard', 'MinMax', 'Log', 또는 None)
        p_degree: 다항식 차수 (None이면 변환 안 함)
        input_data: 변환할 입력 데이터
    """
    if method == 'Standard':
        scaled_data = StandardScaler().fit_transform(input_data)  # z-score 정규화
    elif method == 'MinMax':
        scaled_data = MinMaxScaler().fit_transform(input_data)     # 0~1 정규화
    elif method == 'Log':
        scaled_data = np.log1p(input_data)  # log(1+x): 0값 처리 위해 1을 더함
    else:
        scaled_data = input_data  # 변환 없이 원본 데이터 사용

    # 스케일링 후 추가로 다항식 변환 적용 (선택적)
    if p_degree != None:
        scaled_data = PolynomialFeatures(degree=p_degree,
                                         include_bias=False).fit_transform(scaled_data)
    
    return scaled_data

In [ ]:
# Ridge의 alpha값을 다르게 적용하고 다양한 데이터 변환방법에 따른 RMSE 추출.
alphas = [0.1, 1, 10, 100]

# 변환 방법은 모두 6개, 원본 그대로, 표준정규분포, 표준정규분포+다항식 특성
# 최대/최소 정규화, 최대/최소 정규화+다항식 특성, 로그변환
scale_methods = [(None, None),       # 원본 데이터
                 ('Standard', None), # 표준화만
                 ('Standard', 2),    # 표준화 + 2차 다항식
                 ('MinMax', None),   # MinMax 정규화만
                 ('MinMax', 2),      # MinMax 정규화 + 2차 다항식
                 ('Log', None)]      # 로그 변환만

for scale_method in scale_methods:
    X_data_scaled = get_scaled_data(method=scale_method[0], p_degree=scale_method[1],
                                    input_data=X_data)
    print(X_data_scaled.shape, X_data.shape)  # 다항식 적용 시 피처 수 증가 확인
    print('\n## 변환 유형:{0}, Polynomial Degree:{1}'.format(scale_method[0], scale_method[1]))
    
    # 각 변환 방법별로 Ridge 회귀 성능 평가 (계수 출력은 생략)
    get_linear_reg_eval('Ridge', params=alphas, X_data_n=X_data_scaled,
                        y_target_n=y_target, verbose=False, return_coeff=False)
# 결과 해석:
# 어떤 변환 방법 + alpha 조합이 가장 낮은 RMSE를 보이는지 확인
# 일반적으로 Log 변환이나 StandardScaler가 선형 회귀 성능 향상에 도움